# Multi-Token Prediction

This notebook follows the exact one-stage alignment: main hidden `h_i` plus the teacher-forced embedding of `t_(i+1)` predicts `t_(i+2)`.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'src' / 'model').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Could not find the repository root')
sys.path.insert(0, str(project_root))

import torch

from src.model import DeepSeekConfig, DeepSeekWithMTP, MTPModule, FinalTinyDeepSeek

torch.manual_seed(23)

In [ ]:
config = DeepSeekConfig(
    vocab_size=256,
    max_seq_len=32,
    dim=64,
    n_layers=3,
    n_dense_layers=1,
    n_heads=4,
    q_lora_rank=16,
    kv_lora_rank=8,
    qk_nope_head_dim=8,
    qk_rope_head_dim=4,
    v_head_dim=8,
    inter_dim=128,
    moe_inter_dim=32,
    n_routed_experts=4,
    n_activated_experts=2,
)
model = DeepSeekWithMTP(config, mtp_weight=0.3)
tokens = torch.randint(0, config.vocab_size, (2, 9))
result = model(tokens)

print('main logits:', tuple(result['main_logits'].shape))
print('main hidden:', tuple(result['main_hidden'].shape))
print('MTP logits:', tuple(result['mtp_logits'].shape))
print('main loss:', float(result['main_loss']))
print('MTP loss:', float(result['mtp_loss']))
print('total loss:', float(result['loss']))

For a nine-token input, the main path uses eight positions and the one-stage MTP path uses seven. The main model's embedding table and LM head are passed into the MTP module rather than copied.

In [ ]:
model.zero_grad(set_to_none=True)
result['mtp_loss'].backward()

print('main attention gradient:', model.main_model.layers[0].attention.q_branch.q_down.weight.grad is not None)
print('shared embedding gradient:', model.main_model.embed_tokens.embedding.weight.grad is not None)
print('shared LM head gradient:', model.main_model.lm_head.weight.grad is not None)
print('MTP projection gradient:', model.mtp_module.project.weight.grad is not None)

## Future-token leakage check

Changing the first MTP target `t_2` must not change the first MTP-position logits, because that position only receives `h_0` and the embedding of `t_1`.

In [ ]:
model.eval()
original = torch.tensor([[10, 20, 30, 40, 50, 60]])
changed = original.clone()
changed[:, 2] = 99
with torch.no_grad():
    original_result = model(original)
    changed_result = model(changed)

torch.testing.assert_close(
    original_result['mtp_logits'][:, 0],
    changed_result['mtp_logits'][:, 0],
    rtol=2e-4,
    atol=2e-4,
)
print('first MTP position is causal')